In [43]:
import gymnasium as gym
import ale_py
from gymnasium.wrappers import RecordEpisodeStatistics, RecordVideo, AtariPreprocessing

import numpy as np

import random

from collections import deque

import torch
import torch.nn as nn

import matplotlib.pyplot as plt

In [44]:
training_period = 250 # record the agent's episode every 250

gym.register_envs(ale_py)
env = gym.make("ALE/DoubleDunk-v5", frameskip=1)
env = AtariPreprocessing(env)

In [45]:
device = "cuda" if torch.cuda.is_available() else "cpu"
# device = "cpu"
device

'cuda'

In [46]:
obs, info = env.reset()
obs.shape, info

((84, 84), {'lives': 0, 'episode_frame_number': 52, 'frame_number': 77})

In [47]:
action = env.action_space.sample()
obs, reward, terminated, truncated, info = env.step(action)

In [48]:
reward, info

(0.0, {'lives': 0, 'episode_frame_number': 56, 'frame_number': 81})

In [49]:
action_size = env.action_space.n
action_size

np.int64(18)

In [50]:
conv = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=8, stride=4),
    nn.ReLU(),
    
    nn.Conv2d(32, 64, kernel_size=4, stride=2),
    nn.ReLU(),         
        
    nn.Conv2d(64, 64, kernel_size=3, stride=1),
    nn.ReLU(),
)

obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)

ans = conv(obs_tensor)
ans.shape

torch.Size([64, 7, 7])

In [ ]:
class DDAgent(nn.Module):
    def __init__(self, n_hid, n_out):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),         
               
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(),
        )
        
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, n_hid),
            nn.ReLU(),
            nn.Linear(n_hid, n_out)
        )

    def forward(self, x):
        x = self.conv(x)
        return self.head(x)
    
def choose_action(model, obs, epsilon):
    if np.random.rand() < epsilon:
        return np.random.randint(0, action_size)
    
    obs = torch.tensor(obs, dtype=torch.float32, device=device)
    obs = obs.unsqueeze(0)
    with torch.no_grad():
        out = model(obs)
    return int(torch.argmax(out).item())
    
    
def sample_batch(buffer, batch_size):
    batch = random.sample(buffer, batch_size)

    state, action, reward, next_state, done = map(
        np.array, zip(*batch)
    )

    return (
        torch.tensor(state, dtype=torch.float32, device=device),
        torch.tensor(action, dtype=torch.int64, device=device),
        torch.tensor(reward, dtype=torch.float32, device=device),
        torch.tensor(next_state, dtype=torch.float32, device=device),
        torch.tensor(done, dtype=torch.float32, device=device)
    )

def update_target(target, model):
    target.load_state_dict(model.state_dict())

def train_step(model, target, optimizer, batch, gamma=0.99):
    state, action, reward, next_state, done = batch

    with torch.no_grad():
        next_out = target(next_state).max(dim=1)[0]
        target_out = reward + (1 - done) * gamma * next_out

    out = model(state).gather(1, action.unsqueeze(1)).squeeze(1)

    loss = nn.MSELoss()(out, target_out)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

def train(env, model, target, optimizer,
          step_per_episode=1000,
          episodes=2000,
          batch_size=64, 
          gamma=0.99,
          target_update_freq=1000):
    
    buffer = deque(maxlen=100_000)
    global_step = 0
    reward_history = []

    for episode in range(episodes):
        obs = env.reset()
        steps = 0
        done = False
        total_reward = 0

        if global_step < 30_000:
            epsilon = 0.7
        elif global_step < 100_000:
            epsilon = 0.5
        elif global_step < 150_000:
            epsilon = 0.4
        elif global_step < 200_000:
            epsilon = 0.3
        elif global_step < 250_000:
            epsilon = 0.2
        elif global_step < 300_000:
            epsilon = 0.1
        else:
            epsilon = 0.05
        
        # epsilon = 0.05
            

        while not done and steps < step_per_episode:
            action = choose_action(model, obs, epsilon)
            next_obs, reward, terminated, truncated, info  = env.step(action)
            
            done = terminated or truncated
            
            buffer.append((obs.copy(), action, reward, next_obs.copy(), done))

            obs = next_obs
            total_reward -= 0.1
            total_reward += reward
            global_step += 1
            steps += 1

            if len(buffer) > batch_size:
                batch = sample_batch(buffer, batch_size)
                train_step(model, target, optimizer, batch, gamma)

            if global_step % target_update_freq == 0:
                update_target(target, model)

    
        reward_history.append(total_reward)

        print(f"\rEpisode: {episode + 1} | reward: {total_reward} | eps: {epsilon} | global_steps: {global_step}", end=" ")

    return reward_history

In [52]:
agent = DDAgent(256, action_size)

target = DDAgent(256, action_size)
target.load_state_dict(agent.state_dict())

optimizer = torch.optim.NAdam(agent.parameters(), lr=1e-3)

rewards_history = train(
    env=env,
    model=agent,
    target=target,
    optimizer=optimizer,
    step_per_episode=1000,
    episodes=2000,
    batch_size=64,
    gamma=0.99,
    target_update_freq=1000
)

ValueError: too many values to unpack (expected 3)